# LIBRERIE E DF

In [32]:
import pandas as pd

In [33]:
df_istat = pd.read_csv("../clean_data/ISTAT_clean.csv")
df_situas = pd.read_csv("../clean_data/SITUAS_clean.csv")

In [34]:
print("ISTAT shape:  ", df_istat.shape)
print("SITUAS shape: ",df_situas.shape)

ISTAT shape:   (191184, 5)
SITUAS shape:  (7516, 12)


# MERGE ISTAT-SITUAS

In [35]:
df_merged = df_istat.merge(df_situas, on="IDN_Comune", how="inner")

In [36]:
print("MERGED Shape: ", df_merged.shape)
print(df_merged.columns.tolist())
df_merged.head(3)

MERGED Shape:  (175837, 16)
['IDN_Comune', 'Anno', 'n_Incidenti', 'n_Feriti', 'n_Morti', 'IDA_Comune', 'Comune', 'ID_Ripartizione_Geografica', 'ID_Regione', 'ID_Provincia/Uts', 'Sigla_Auto', 'Superficie_Kmq', 'Pop_Residente', 'IS_Capoluogo_Provincia/Uts', 'IS_Capoluogo_Regione', 'Anno_Rif_SITUAS']


,IDN_Comune,Anno,n_Incidenti,n_Feriti,n_Morti,IDA_Comune,Comune,ID_Ripartizione_Geografica,ID_Regione,ID_Provincia/Uts,Sigla_Auto,Superficie_Kmq,Pop_Residente,IS_Capoluogo_Provincia/Uts,IS_Capoluogo_Regione,Anno_Rif_SITUAS
0,1001,2001,5,10,0,1001,Agliè,1,1,201,TO,13.1463,2585,0,0,2024
1,1001,2002,5,10,0,1001,Agliè,1,1,201,TO,13.1463,2585,0,0,2024
2,1001,2003,4,7,0,1001,Agliè,1,1,201,TO,13.1463,2585,0,0,2024


# KPI INCIDENTI e OUTLIERS
incidentalita' per popolazione = (tot incidenti / Pop residente) * 1.000

incidentalita' per kmq = (tot incidenti / superfice kmq)

indice lesivita' = TOT FERITI / tot incidenti

indice mortalita' = TOT MORTI / tot incidenti

gravita' incidenti = (TOT MORTI/ tot incidenti) * 100


In [37]:
# INCIDENTALITA' X POP (quanti incidenti ogni 1000 abitanti)
df_merged["Incidentalita_x_pop"] = (df_merged["n_Incidenti"] / df_merged["Pop_Residente"] * 1000).round(2)

# INCIDENTALITA' PER KMQ (quanti incidenti per kmq)
df_merged["Incidentalita_x_kmq"] = (df_merged["n_Incidenti"] / df_merged["Superficie_Kmq"]).round(2)

# INDICE LESIVITA' (quanti feriti ogni 100 incidenti)
df_merged["Indice_Lesivita"] = df_merged.apply(
    lambda riga: 0 if riga["n_Incidenti"] == 0 
                   else round(riga["n_Feriti"] / riga["n_Incidenti"] * 100, 2),
    axis=1)

# INDICE GRAVITA' (quanti morti ogni 100 incidenti)
df_merged["Indice_Gravita"] = df_merged.apply(
    lambda riga: 0 if riga["n_Incidenti"] == 0 
                   else round(riga["n_Morti"] / riga["n_Incidenti"] * 100, 2),
    axis=1)

In [39]:
print(df_merged.dtypes)

IDN_Comune                      int64
Anno                            int64
n_Incidenti                     int64
n_Feriti                        int64
n_Morti                         int64
IDA_Comune                      int64
Comune                            str
ID_Ripartizione_Geografica      int64
ID_Regione                      int64
ID_Provincia/Uts                int64
Sigla_Auto                        str
Superficie_Kmq                float64
Pop_Residente                   int64
IS_Capoluogo_Provincia/Uts      int64
IS_Capoluogo_Regione            int64
Anno_Rif_SITUAS                 int64
Incidentalita_x_pop           float64
Incidentalita_x_kmq           float64
Indice_Lesivita               float64
Indice_Gravita                float64
dtype: object


In [43]:
df_merged[["Incidentalita_x_pop","Incidentalita_x_kmq", "Indice_Lesivita", "Indice_Gravita"]].describe()

,Incidentalita_x_pop,Incidentalita_x_kmq,Indice_Lesivita,Indice_Gravita
count,175837.000000,175837.000000,175837.000000,175837.000000
mean,2.062574,0.736583,118.446378,4.047790
std,2.518354,2.077018,80.013073,14.383973
min,0.000000,0.000000,0.000000,0.000000
25%,0.490000,0.030000,100.000000,0.000000
50%,1.570000,0.170000,128.570000,0.000000
75%,2.860000,0.610000,159.890000,0.000000
max,241.500000,99.130000,2500.000000,500.000000


In [64]:
q99_grav = df_merged["Indice_Gravita"].quantile(0.99)
q99_les  = df_merged["Indice_Lesivita"].quantile(0.99)

outliers99 = df_merged[(df_merged["Indice_Gravita"] > q99_grav) | (df_merged["Indice_Lesivita"] > q99_les)]

print("comuni totali: ", len(df_merged))
print("Comuni outliers (oltre 99° percentile): ", len(outliers99))

comuni totali:  175837
Comuni outliers (oltre 99° percentile):  1919


In [63]:
outliers99[["Comune","Anno","Incidentalita_x_pop","Incidentalita_x_kmq", "Indice_Lesivita", "Indice_Gravita"]]

,Comune,Anno,Incidentalita_x_pop,Incidentalita_x_kmq,Indice_Lesivita,Indice_Gravita
56,Ala di Stura,2009,2.16,0.02,400.00,0.0
83,Albiano d'Ivrea,2012,1.23,0.17,350.00,0.0
129,Alpette,2010,8.06,0.36,350.00,0.0
135,Alpette,2016,4.03,0.18,500.00,0.0
185,Andezeno,2019,0.50,0.13,400.00,0.0
...,...,...,...,...,...,...
173073,Craveggia,2004,1.25,0.03,600.00,0.0
173276,Gignese,2016,0.92,0.07,500.00,0.0
173433,Malesco,2010,2.25,0.07,433.33,0.0
173593,Nonio,2005,2.41,0.20,400.00,0.0


# EXPORT DEL CSV

# df_merged.to_csv("../exports/dataset_finale.csv", index=False)